# **Preparacion del ambiente**

In [38]:
# Recursos necesarios
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import zipfile
import os
import pandas as pd

In [5]:
# Kaggle
!mkdir ~/.kaggle

In [6]:
!chmod 600 ~/.kaggle/kaggle.json

In [7]:
# vamos a descargar el dataset
! kaggle datasets download megelon/meetup --force

Dataset URL: https://www.kaggle.com/datasets/megelon/meetup
License(s): unknown
 55% 108M/197M [00:00<00:00, 1.13GB/s]
100% 197M/197M [00:00<00:00, 689MB/s] 


In [4]:
#Iniciar conexión de spark
spark = SparkSession.builder\
        .master("local")\
        .appName("Colab")\
        .config('spark.ui.port', '4050')\
        .getOrCreate()

In [8]:
for file in os. listdir():
    if file.endswith('.zip'):
      zip_ref = zipfile.ZipFile(file, 'r')
      zip_ref.extractall()
      zip_ref.close()

## **Leer los datos_groups**

In [9]:
df = spark.read.csv('/content/groups.csv', header=True)

In [10]:
print(df.count(),len(df.columns))

16330 36


In [11]:
from pyspark.sql.functions import col, count, lit

# Análisis rápido en PySpark
print(f"Total filas: {df.count()}")
print(f"Total columnas: {len(df.columns)}")


Total filas: 16330
Total columnas: 36


## **Limpieza y preparacion datos**

In [ ]:
# Leer solo el encabezado para ver nombres reales
real_headers = spark.read.csv("groups.csv", header=True).limit(0)
print("Columnas reales en el CSV:", real_headers.columns)

# Leer muestra de datos sin esquema
sample_df = spark.read.csv("groups.csv", header=True).limit(5)
sample_df.show(truncate=False)

In [12]:
df.printSchema()

root
 |-- group_id: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- category.name: string (nullable = true)
 |-- category.shortname: string (nullable = true)
 |-- city_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- created: string (nullable = true)
 |-- description: string (nullable = true)
 |-- group_photo.base_url: string (nullable = true)
 |-- group_photo.highres_link: string (nullable = true)
 |-- group_photo.photo_id: string (nullable = true)
 |-- group_photo.photo_link: string (nullable = true)
 |-- group_photo.thumb_link: string (nullable = true)
 |-- group_photo.type: string (nullable = true)
 |-- join_mode: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- link: string (nullable = true)
 |-- lon: string (nullable = true)
 |-- members: string (nullable = true)
 |-- group_name: string (nullable = true)
 |-- organizer.member_id: string (nullable = true)
 |-- organizer.name: strin

In [20]:
# Contar los registros de cada columna
total = df.count()

# Crear las expresiones de conteo y porcentaje
# Use backticks to escape column names with special characters
non_null_exprs = [count(col(f"`{c}`")).alias(f"{c}_non_null") for c in df.columns]
percent_exprs = [(count(col(f"`{c}`"))/lit(total)*100).alias(f"{c}_percent") for c in df.columns]

# Combinar las expresiones y aplicar select
df.select(*non_null_exprs, *percent_exprs).show(vertical=True)

-RECORD 0--------------------------------------------------
 group_id_non_null                     | 16330             
 category_id_non_null                  | 16330             
 category.name_non_null                | 16330             
 category.shortname_non_null           | 16330             
 city_id_non_null                      | 16330             
 city_non_null                         | 16330             
 country_non_null                      | 16330             
 created_non_null                      | 16330             
 description_non_null                  | 16328             
 group_photo.base_url_non_null         | 16330             
 group_photo.highres_link_non_null     | 16330             
 group_photo.photo_id_non_null         | 16330             
 group_photo.photo_link_non_null       | 16330             
 group_photo.thumb_link_non_null       | 16330             
 group_photo.type_non_null             | 16330             
 join_mode_non_null                    |

## **Estructura gruops**

In [21]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.functions import regexp_replace
from pyspark.sql.functions import from_json
from pyspark.sql.functions import col, count

# Definir el esquema para optimizar la lectura
groups_schema = StructType([
    StructField("group_id", IntegerType(), True),  # Cambiar a IntegerType
    StructField("category.shortname", StringType(), True),
    StructField("city", StringType(), True),
    StructField("country", StringType(), True),
    StructField("created", TimestampType(), True),  # Cambiar a TimestampType
    StructField("join_mode", StringType(), True),
    StructField("lat", DoubleType(), True),  # Cambiar a DoubleType
    StructField("lon", DoubleType(), True),  # Cambiar a DoubleType
    StructField("members", IntegerType(), True),  # Cambiar a IntegerType
    StructField("rating", DoubleType(), True),  # Cambiar a DoubleType
    StructField("urlname", StringType(), True),
    StructField("group_topics", StringType(), True),
    StructField("visibility", StringType(), True),
    StructField("who", StringType(), True)
])

# Leer el archivo CSV
df_groups = spark.read \
    .option("header", "true") \
    .option("delimiter", ",") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .option("mode", "PERMISSIVE") \
    .option("nullValue", "null") \
    .option("nanValue", "null") \
    .csv("groups.csv")

# Verificar qué datos llegaron
df_groups.select("lat", "lon", "members", "rating", "created").limit(5).show()

# Define a function to handle column names with special characters
def safe_col_name(name):
    return f"`{name}`"

# Access the column with the correct name
count_df = df_groups.select([count(col(safe_col_name(c))).alias(c) for c in df_groups.columns])
count_df.show(vertical=True)


+-----------+------------+-------+------+-------------------+
|        lat|         lon|members|rating|            created|
+-----------+------------+-------+------+-------------------+
|40.75000000|-73.98999800|   1440|  4.39|2002-11-21 16:50:46|
|40.75000000|-73.98999800|    969|  4.31|2003-05-20 14:48:54|
|40.73000000|-73.98999800|   2930|  4.84|2004-03-27 09:55:41|
|40.75000000|-73.98999800|   5080|  4.46|2002-11-16 04:49:16|
|40.72000100|-74.00000000|   2097|  4.09|2003-10-22 21:39:49|
+-----------+------------+-------+------+-------------------+

-RECORD 0-----------------------------
 group_id                     | 16330 
 category_id                  | 16330 
 category.name                | 16330 
 category.shortname           | 16330 
 city_id                      | 16330 
 city                         | 16330 
 country                      | 16330 
 created                      | 16330 
 description                  | 16330 
 group_photo.base_url         | 16330 
 group_photo

##**Procesamiento**

In [22]:
# Seleccionar y transformar las columnas necesarias
df_groups_light = df_groups.select(
    col("group_id").cast("long").alias("group_id"),
    col("`category.shortname`").alias("group_name"), # Use backticks to escape the column name
    col("city").alias("group_city"),
    col("country").alias("group_country"),
    to_timestamp(col("created"), "yyyy-MM-dd HH:mm:ss").alias("group_created"),
    col("join_mode").alias("group_join_mode"),
    regexp_replace(col("lat"), ",", ".").cast("float").alias("group_lat"),
    regexp_replace(col("lon"), ",", ".").cast("float").alias("group_lon"),
    col("members").cast("integer").alias("group_members"),
    col("rating").cast("float").alias("group_rating"),
    col("urlname").alias("group_urlname"),
    #lit("[]").alias("group_topics_json"),  # Array JSON vacío
    col("visibility").alias("group_visibility"),
    col("who").alias("group_who"),
    current_timestamp().alias("load_timestamp")
)

print(df_groups_light.count(), len(df_groups_light.columns))

# Filtrar datos para hacerlo más liviano
# Por ejemplo, solo grupos con más de 100 miembros
df_groups_p = df_groups_light.filter(col("group_members") > 1000)
print(df_groups_p.count(), len(df_groups_p.columns))

df_groups_p.show(5)

16330 14
2884 14
+--------+----------------+----------+-------------+-------------------+---------------+---------+---------+-------------+------------+--------------------+----------------+--------------------+--------------------+
|group_id|      group_name|group_city|group_country|      group_created|group_join_mode|group_lat|group_lon|group_members|group_rating|       group_urlname|group_visibility|           group_who|      load_timestamp|
+--------+----------------+----------+-------------+-------------------+---------------+---------+---------+-------------+------------+--------------------+----------------+--------------------+--------------------+
|    6388|health-wellbeing|  New York|           US|2002-11-21 16:50:46|           open|    40.75|   -73.99|         1440|        4.39|alternative-healt...|          public| Explorers of Health|2025-04-29 08:48:...|
|    8458|    pets-animals|  New York|           US|2004-03-27 09:55:41|           open|    40.73|   -73.99|         29

**Otras validaciones**

In [23]:
import chardet

with open("groups.csv", "rb") as f:
    result = chardet.detect(f.read())

print(result)

{'encoding': 'ascii', 'confidence': 1.0, 'language': ''}


In [17]:
# Definir esquema para los topics
topics_schema = ArrayType(StructType([
    StructField("topic_name", StringType()),
    StructField("urlkey", StringType())
]))

# Parsear el JSON
df_with_topics = df_groups_light.withColumn(
    "group_topics",
    from_json("group_topics_json", topics_schema)
)

##**GROUPS**

In [51]:
# 1. Guardar el DataFrame como CSV en el sistema de archivos local
df_groups_p.write.csv("meetup_groups_processed", header=True, mode="overwrite")

# 2. Comprimir los resultados (PySpark genera múltiples archivos)
!zip -r meetup_data2.zip meetup_groups_processed/

# 3. Descargar el archivo comprimido (en Colab)
from google.colab import files
files.download("meetup_data2.zip")


updating: meetup_groups_processed/ (stored 0%)
updating: meetup_groups_processed/_SUCCESS (stored 0%)
updating: meetup_groups_processed/._SUCCESS.crc (stored 0%)
  adding: meetup_groups_processed/.part-00000-e53dab2c-5bd3-44b1-b338-35d2754705e2-c000.csv.crc (stored 0%)
  adding: meetup_groups_processed/part-00000-e53dab2c-5bd3-44b1-b338-35d2754705e2-c000.csv (deflated 75%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##**Leer datos Eventos**

In [24]:
df2 = spark.read.csv('/content/events.csv', header=True)

In [25]:
# Leer muestra de datos sin esquema
sample_df2 = spark.read.csv("events.csv", header=True).limit(5)
sample_df2.show(truncate=False)

+---------+-------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [26]:
# imprimir el esquema de events.csv

df2.printSchema()


root
 |-- event_id: string (nullable = true)
 |-- created: string (nullable = true)
 |-- description: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- event_url: string (nullable = true)
 |-- fee.accepts: string (nullable = true)
 |-- fee.amount: string (nullable = true)
 |-- fee.currency: string (nullable = true)
 |-- fee.description: string (nullable = true)
 |-- fee.label: string (nullable = true)
 |-- fee.required: string (nullable = true)
 |-- group.created: string (nullable = true)
 |-- group.group_lat: string (nullable = true)
 |-- group.group_lon: string (nullable = true)
 |-- group_id: string (nullable = true)
 |-- group.join_mode: string (nullable = true)
 |-- group.name: string (nullable = true)
 |-- group.urlname: string (nullable = true)
 |-- group.who: string (nullable = true)
 |-- headcount: string (nullable = true)
 |-- how_to_find_us: string (nullable = true)
 |-- maybe_rsvp_count: string (nullable = true)
 |-- event_name: string (nullable = true)


##**Estrutura de datos Eventos**

In [27]:
event_schema = StructType([
    StructField("event_id", IntegerType(), True),
    StructField("event_name", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("event_url", StringType(), True),
    StructField("event_description", StringType(), True),
    StructField("group_id", LongType(), True),
    StructField("group_name", StringType(), True),
    StructField("group_country", StringType(), True),
    StructField("venue_id", LongType(), True),
    StructField("venue_name", StringType(), True),
    StructField("lat", DoubleType(), True),  # Cambiar a DoubleType
    StructField("lon", DoubleType(), True),  # Cambiar a DoubleType
    StructField("venue_address_1", StringType(), True),
    StructField("venue_city", StringType(), True),
    StructField("venue_country", StringType(), True),
    StructField("yes_rsvp_count", LongType(), True),
    StructField("waitlist_count", LongType(), True),
    StructField("created", TimestampType(), True),
    StructField("updated", TimestampType(), True),
    StructField("duration", LongType(), True),
    StructField("load_timestamp", TimestampType(), True)
])

# Leer el archivo CSV
df_events = spark.read \
    .option("header", "true") \
    .option("delimiter", ",") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .option("mode", "PERMISSIVE") \
    .option("nullValue", "null") \
    .option("nanValue", "null") \
    .csv("events.csv")

# Verificar qué datos llegaron
df_events.select("`venue.lat`", "`venue.lon`", "group_id", "venue_id", "created").limit(5).show()

+-----------+-------------+--------+--------+-------------------+
|  venue.lat|    venue.lon|group_id|venue_id|            created|
+-----------+-------------+--------+--------+-------------------+
|37.79795000|-122.40569000| 5817262|23729697|2013-12-03 21:24:29|
|37.79817200|-122.40545700| 1627081|16948982|2014-05-20 18:52:00|
|37.79751600|-122.40739400| 1627081|24717469|2014-10-23 16:18:44|
|37.79803600|-122.40544200| 5817262|  724783|2015-02-28 19:27:32|
|37.79805000|-122.40525100| 1627081|20984572|2016-01-08 21:35:40|
+-----------+-------------+--------+--------+-------------------+



##**Eventos**

In [47]:
# Procesar datos para eventos
df_events_p = df_events.select(
    # Mapeo directo de campos
    col("event_id").cast("long").alias("event_id"),
    col("event_name"),
    to_timestamp(col("event_time")).alias("event_time"),  # Convertir de string a timestamp
    col("event_url"),
    #col("description").alias("event_description"),  # Mapeo de nombre diferente
    col("group_id").cast("long").alias("group_id"),
    col("`group.name`").alias("group_name"),
    #col("`venue.localized_country_name`").alias("group_country"),  # Ojo con el typo en country
    #col("venue_id").cast("long").alias("venue_id"),
    col("`venue.name`").alias("venue_name"),
    #col("`venue.lat`").cast("double").alias("venue_lat"),
    #col("`venue.lon`").cast("double").alias("venue_lon"),
    col("`venue.address_1`").alias("venue_address"),
    col("`venue.city`").alias("venue_city"),
    col("`venue.country`").alias("venue_country"),
    col("yes_rsvp_count").cast("long").alias("yes_rsvp_count"),
    col("waitlist_count").cast("long").alias("waitlist_count"),
    to_timestamp(col("created")).alias("created"),
    to_timestamp(col("updated")).alias("updated"),
    round(col("duration")/(60*60), 2).cast("long").alias("duration"),
    current_timestamp().alias("load_timestamp")
)

print(df_events_p.count(), len(df_events_p.columns))
df_events_p.show(5)

# Contar todos los registros
total = df_events_p.count()
print(f"Total registros: {total}")

# Contar no nulos en una columna
#no_nulos = df_events_p.select("columna").na.drop().count()
#print(f"Registros no nulos: {no_nulos}")

5807 16
+---------+--------------------+-------------------+--------------------+--------+--------------------+--------------------+----------------+-------------+-------------+--------------+--------------+-------------------+-------------------+--------+--------------------+
| event_id|          event_name|         event_time|           event_url|group_id|          group_name|          venue_name|   venue_address|   venue_city|venue_country|yes_rsvp_count|waitlist_count|            created|            updated|duration|      load_timestamp|
+---------+--------------------+-------------------+--------------------+--------+--------------------+--------------------+----------------+-------------+-------------+--------------+--------------+-------------------+-------------------+--------+--------------------+
|153868222|Murder Mystery Di...|2017-10-30 02:30:00|https://www.meetu...| 5817262|San Francisco Sta...|     Little Szechuan|505 Broadway St.|San Francisco|           us|            7

In [48]:
df_events_p.printSchema()

root
 |-- event_id: long (nullable = true)
 |-- event_name: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- event_url: string (nullable = true)
 |-- group_id: long (nullable = true)
 |-- group_name: string (nullable = true)
 |-- venue_name: string (nullable = true)
 |-- venue_address: string (nullable = true)
 |-- venue_city: string (nullable = true)
 |-- venue_country: string (nullable = true)
 |-- yes_rsvp_count: long (nullable = true)
 |-- waitlist_count: long (nullable = true)
 |-- created: timestamp (nullable = true)
 |-- updated: timestamp (nullable = true)
 |-- duration: long (nullable = true)
 |-- load_timestamp: timestamp (nullable = false)



**Otros**

In [68]:
# 1. Guardar el DataFrame como CSV en el sistema de archivos local
df_events_p.write.csv("meetup_events_processed", header=True, mode="overwrite")

# 2. Comprimir los resultados (PySpark genera múltiples archivos)
!zip -r meetup_data.zip meetup_events_processed/

# 3. Descargar el archivo comprimido (en Colab)
from google.colab import files
files.download("meetup_data.zip")

updating: meetup_events_processed/ (stored 0%)
updating: meetup_events_processed/_SUCCESS (stored 0%)
updating: meetup_events_processed/._SUCCESS.crc (stored 0%)
  adding: meetup_events_processed/.part-00000-ce1131c2-6a1c-4392-9fa7-ea8178824ede-c000.csv.crc (stored 0%)
  adding: meetup_events_processed/part-00000-ce1131c2-6a1c-4392-9fa7-ea8178824ede-c000.csv (deflated 93%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [61]:
df_joined_ge = df_groups_p.join(
    df_events_p,
    df_groups_p["group_id"] == df_events_p["group_id"],
    "inner"
).select(
    col("event_id").alias("event_id_original"),  # Forzar nombre único
    col("event_name"),
    df_groups_p["group_id"].alias("group_id"),  # Specify the source DataFrame using df_group_light_p
    df_groups_p["group_name"].alias("group_name") # Specify the source DataFrame using df_group_light_p
)

df_joined_ge.show(5)

+-----------------+--------------------+--------+---------------+
|event_id_original|          event_name|group_id|     group_name|
+-----------------+--------------------+--------+---------------+
|             NULL|NYC Creative and ...|   54691|career-business|
|             NULL|NYC Creative and ...|   54691|career-business|
|             NULL|NYC Creative and ...|   54691|career-business|
|             NULL|NYC Creative and ...|   54691|career-business|
|             NULL|NYC Creative and ...|   54691|career-business|
+-----------------+--------------------+--------+---------------+
only showing top 5 rows



##**Miembros**

In [55]:
df3 = spark.read.csv('/content/members.csv', header=True)

In [58]:
# Leer muestra de datos sin esquema
sample_df3 = spark.read.csv("members.csv", header=True).limit(5)
sample_df3.show(truncate=False)

+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+-------+------------+-------------------+-----------+-------------------------------+------------+-----------+-----+-------------+-------------------+--------+
|member_id|bio                                                                                                                                                                                       |city    |country|hometown    |joined             |lat        |link                           |lon         |member_name|state|member_status|visited            |group_id|
+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+-------+------------+-------------------+-----------+------

In [93]:
df3.printSchema()

root
 |-- member_id: string (nullable = true)
 |-- bio: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- hometown: string (nullable = true)
 |-- joined: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- link: string (nullable = true)
 |-- lon: string (nullable = true)
 |-- member_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- member_status: string (nullable = true)
 |-- visited: string (nullable = true)
 |-- group_id: string (nullable = true)



##**Estrucutura Miembros**

In [74]:
members_schema = StructType([
    StructField("member_id", IntegerType(), True),
    #StructField("bio", StringType(), True),
    StructField("city", StringType(), True),
    StructField("country", StringType(), True),
    StructField("hometown", StringType(), True),
    StructField("joined", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("link", StringType(), True),
    StructField("lon", DoubleType(), True),
    StructField("member_name", StringType(), True),
    StructField("state", StringType(), True),
    StructField("member_status", StringType(), True),
    StructField("visited", TimestampType(), True),
    StructField("group_id", LongType(), True),
    StructField("load_timestamp", TimestampType(), False)
])

# Leer el archivo CSV
df_members = spark.read \
    .option("header", "true") \
    .option("delimiter", ",") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .option("mode", "PERMISSIVE") \
    .option("nullValue", "null") \
    .option("nanValue", "null") \
    .csv("members.csv")

# Verificar qué datos llegaron
df_members.select("lat", "lon", "group_id", "visited", "visited").limit(5).show()

+-----------+------------+--------+-------------------+-------------------+
|        lat|         lon|group_id|            visited|            visited|
+-----------+------------+--------+-------------------+-------------------+
|40.72000000|-74.00000000|  490552|2009-09-18 18:32:23|2009-09-18 18:32:23|
|40.72000000|-74.00000000| 1474611|2011-03-20 01:02:11|2011-03-20 01:02:11|
|40.72000000|-74.00000000| 1490492|2011-01-18 20:37:23|2011-01-18 20:37:23|
|40.72000000|-74.00000000| 1515830|2011-07-23 03:42:28|2011-07-23 03:42:28|
|40.72000000|-74.00000000| 1574965|2011-06-13 18:33:23|2011-06-13 18:33:23|
+-----------+------------+--------+-------------------+-------------------+



In [75]:
# Procesar datos para miembros
df_members_p = df_members.select(
    col("member_id").cast("long").alias("member_id"),
    #col("bio"),
    col("city").alias("member_city"),
    col("country").alias("member_country"),
    col("hometown").alias("member_hometown"),
    to_timestamp(col("joined")).alias("member_joined"),
    col("lat").cast("double").alias("member_lat"),
    col("link").alias("member_link"),
    col("lon").cast("double").alias("member_lon"),
    col("member_name"),
    col("state").alias("member_state"),
    col("member_status"),
    to_timestamp(col("visited")).alias("member_visited"),
    col("group_id").cast("long").alias("group_id"),
    current_timestamp().alias("load_timestamp")
)

print(df_members_p.count(), len(df_members_p.columns))
df_members_p.show(5)

5893886 14
+---------+-----------+--------------+---------------+-------------------+----------+--------------------+----------+-----------+------------+-------------+-------------------+--------+--------------------+
|member_id|member_city|member_country|member_hometown|      member_joined|member_lat|         member_link|member_lon|member_name|member_state|member_status|     member_visited|group_id|      load_timestamp|
+---------+-----------+--------------+---------------+-------------------+----------+--------------------+----------+-----------+------------+-------------+-------------------+--------+--------------------+
|        3|   New York|            us|   New York, NY|2007-05-01 22:04:37|     40.72|http://www.meetup...|     -74.0|Matt Meeker|          NY|       active|2009-09-18 18:32:23|  490552|2025-04-29 12:21:...|
|        3|   New York|            us|   New York, NY|2011-01-23 14:13:17|     40.72|http://www.meetup...|     -74.0|Matt Meeker|          NY|       active|2011-

In [76]:
# 1. Guardar el DataFrame como CSV en el sistema de archivos local
df_events_p.write.csv("meetup_members_processed", header=True, mode="overwrite")

# 2. Comprimir los resultados (PySpark genera múltiples archivos)
!zip -r meetup_data.zip meetup_members_processed/

# 3. Descargar el archivo comprimido (en Colab)
from google.colab import files
files.download("meetup_data.zip")

updating: meetup_members_processed/ (stored 0%)
updating: meetup_members_processed/_SUCCESS (stored 0%)
updating: meetup_members_processed/._SUCCESS.crc (stored 0%)
  adding: meetup_members_processed/part-00000-b72df11a-6be0-4bdd-95ad-dcb65bb449f3-c000.csv (deflated 93%)
  adding: meetup_members_processed/.part-00000-b72df11a-6be0-4bdd-95ad-dcb65bb449f3-c000.csv.crc (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [77]:
df_members_p.printSchema()

root
 |-- member_id: long (nullable = true)
 |-- member_city: string (nullable = true)
 |-- member_country: string (nullable = true)
 |-- member_hometown: string (nullable = true)
 |-- member_joined: timestamp (nullable = true)
 |-- member_lat: double (nullable = true)
 |-- member_link: string (nullable = true)
 |-- member_lon: double (nullable = true)
 |-- member_name: string (nullable = true)
 |-- member_state: string (nullable = true)
 |-- member_status: string (nullable = true)
 |-- member_visited: timestamp (nullable = true)
 |-- group_id: long (nullable = true)
 |-- load_timestamp: timestamp (nullable = false)



In [ ]:
df_joined_ge = df_groups_p.join(
    df_events_p,
    df_groups_p["group_id"] == df_members_p["group_id"],
    "inner"
).select(
    col("event_id").alias("event_id_original"),  # Forzar nombre único
    col("event_name"),
    df_groups_p["group_id"].alias("group_id"),  # Specify the source DataFrame using df_group_light_p
    df_groups_p["group_name"].alias("group_name") # Specify the source DataFrame using df_group_light_p
)

df_joined_ge.show(5)

In [85]:
# seleccionar solo los grupos de este experimento
df_joined_k = df_groups_p.alias("g").join(
    df_members_p.alias("m"),
    df_groups_p["group_id"] == df_members_p["group_id"],
    "inner"
).select(
    *[col("m." + c) for c in df_members_p.columns if c != "group_id"],
    col("g.group_id")
)

df_joined_k.show(5)
total_registros = df_joined_k.count()
print(f"Total de registros después del JOIN: {total_registros}")


+---------+-----------+--------------+---------------+-------------------+----------+--------------------+----------+-----------+------------+-------------+-------------------+--------------------+--------+
|member_id|member_city|member_country|member_hometown|      member_joined|member_lat|         member_link|member_lon|member_name|member_state|member_status|     member_visited|      load_timestamp|group_id|
+---------+-----------+--------------+---------------+-------------------+----------+--------------------+----------+-----------+------------+-------------+-------------------+--------------------+--------+
|        3|   New York|            us|   New York, NY|2007-05-01 22:04:37|     40.72|http://www.meetup...|     -74.0|Matt Meeker|          NY|       active|2009-09-18 18:32:23|2025-04-29 12:50:...|  490552|
|        3|   New York|            us|   New York, NY|2011-01-23 14:13:17|     40.72|http://www.meetup...|     -74.0|Matt Meeker|          NY|       active|2011-03-20 01:02

In [88]:
# Verificar nulos en columnas críticas
critical_columns = ["group_id", "member_id"]

df_joined_k.select(
    [count(when(col(c).isNull(), c)).alias(f"null_{c}") for c in critical_columns]
).show()

+-------------+--------------+
|null_group_id|null_member_id|
+-------------+--------------+
|            0|             0|
+-------------+--------------+



In [ ]:
# Análisis completo de calidad de datos
def check_data_quality(df):
    from pyspark.sql.functions import lit

    quality_stats = []
    for column in df_joined_k.columns:
        total = df_joined_k.count()
        null_count = df_joined_k.filter(col(column).isNull()).count()
        distinct_count = df_joined_k.select(column).distinct().count()

        quality_stats.append({
            "columna": column,
            "total_nulos": null_count,
            "porcentaje_nulos": (null_count/total)*100,
            "valores_unicos": distinct_count
        })

    return spark.createDataFrame(quality_stats)

# Ejecutar el análisis
quality_report = check_data_quality(df_joined_k)
quality_report.show(truncate=False)

In [92]:
# 1. Guardar el DataFrame como CSV en el sistema de archivos local
df_joined_k.write.csv("meetup_members_processed", header=True, mode="overwrite")

# 2. Comprimir los resultados (PySpark genera múltiples archivos)
!zip -r meetup_data.zip meetup_members_processed/

# 3. Descargar el archivo comprimido (en Colab)
from google.colab import files
files.download("meetup_data.zip")

updating: meetup_members_processed/ (stored 0%)
updating: meetup_members_processed/_SUCCESS (stored 0%)
updating: meetup_members_processed/._SUCCESS.crc (stored 0%)
  adding: meetup_members_processed/.part-00009-4f714607-0bb7-4e84-9970-2eaad0374544-c000.csv.crc (deflated 0%)
  adding: meetup_members_processed/part-00002-4f714607-0bb7-4e84-9970-2eaad0374544-c000.csv (deflated 86%)
  adding: meetup_members_processed/part-00001-4f714607-0bb7-4e84-9970-2eaad0374544-c000.csv (deflated 86%)
  adding: meetup_members_processed/.part-00007-4f714607-0bb7-4e84-9970-2eaad0374544-c000.csv.crc (deflated 0%)
  adding: meetup_members_processed/.part-00000-4f714607-0bb7-4e84-9970-2eaad0374544-c000.csv.crc (deflated 0%)
  adding: meetup_members_processed/part-00008-4f714607-0bb7-4e84-9970-2eaad0374544-c000.csv (deflated 89%)
  adding: meetup_members_processed/.part-00008-4f714607-0bb7-4e84-9970-2eaad0374544-c000.csv.crc (deflated 0%)
  adding: meetup_members_processed/part-00000-4f714607-0bb7-4e84-9970-

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [91]:
df_joined_k.printSchema()

root
 |-- member_id: long (nullable = true)
 |-- member_city: string (nullable = true)
 |-- member_country: string (nullable = true)
 |-- member_hometown: string (nullable = true)
 |-- member_joined: timestamp (nullable = true)
 |-- member_lat: double (nullable = true)
 |-- member_link: string (nullable = true)
 |-- member_lon: double (nullable = true)
 |-- member_name: string (nullable = true)
 |-- member_state: string (nullable = true)
 |-- member_status: string (nullable = true)
 |-- member_visited: timestamp (nullable = true)
 |-- load_timestamp: timestamp (nullable = false)
 |-- group_id: long (nullable = true)



##**LECTURA DATOS LUGARES**

In [94]:
df3 = spark.read.csv('/content/venues.csv', header=True)

# Leer muestra de datos sin esquema
sample_df3 = spark.read.csv("venues.csv", header=True).limit(5)
sample_df3.show(truncate=False)

df3.printSchema()

+--------+-------------------+--------+-------+--------+-----------+----------------------+------------+-----------------------+------+------------+-----+-----+-----------------+
|venue_id|address_1          |city    |country|distance|lat        |localized_country_name|lon         |venue_name             |rating|rating_count|state|zip  |normalised_rating|
+--------+-------------------+--------+-------+--------+-----------+----------------------+------------+-----------------------+------+------------+-----+-----+-----------------+
|5286    |424 Park Ave S     |New York|us     |0.00    |40.74425900|USA                   |-73.98374900|Starbucks Coffee       |4.00  |51.00       |NY   |10016|3.92             |
|5293    |Union Square       |New York|us     |0.00    |40.73139000|USA                   |-73.98840000|Virgin Megastore (cafe)|2.83  |109.00      |NY   |10003|2.80             |
|8356    |141 West 72nd St.  |New York|us     |0.00    |40.77827500|USA                   |-73.98009500|K

In [100]:
# Estrucutura Lugares
venues_schema = StructType([
    StructField("venue_id", IntegerType(), False),  # NOT NULL
    StructField("address_1", StringType(), True),
    StructField("city", StringType(), True),
    #StructField("state", StringType(), True),
    #StructField("zip", StringType(), True),
    StructField("country", StringType(), True),
    #StructField("localized_country_name", StringType(), True),
    #StructField("lat", DoubleType(), True),
    #StructField("lon", DoubleType(), True),
    #StructField("distance", DoubleType(), True),
    StructField("venue_name", StringType(), True),
    # Ratings y valoraciones (convertidos a tipos numéricos)
    #StructField("rating", DoubleType(), True),
    StructField("rating_count", IntegerType(), True),
    StructField("normalised_rating", DoubleType(), True),
    StructField("load_timestamp", TimestampType(), False)
])

# Leer el archivo CSV
df_venues = spark.read \
    .option("header", "true") \
    .option("delimiter", ",") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .option("mode", "PERMISSIVE") \
    .option("nullValue", "null") \
    .option("nanValue", "null") \
    .csv("venues.csv")

# Add load_timestamp column after reading the CSV
df_venues = df_venues.withColumn("load_timestamp", current_timestamp())

# Verificar qué datos llegaron
df_venues.select("venue_id", "address_1", "rating_count", "normalised_rating", "load_timestamp").limit(5).show()

+--------+-------------------+------------+-----------------+--------------------+
|venue_id|          address_1|rating_count|normalised_rating|      load_timestamp|
+--------+-------------------+------------+-----------------+--------------------+
|    5286|     424 Park Ave S|       51.00|             3.92|2025-04-29 14:41:...|
|    5293|       Union Square|      109.00|             2.80|2025-04-29 14:41:...|
|    8356|  141 West 72nd St.|       18.00|             2.00|2025-04-29 14:41:...|
|   11898|7000 North Glenwood|      275.00|             3.79|2025-04-29 14:41:...|
|   12152|Madison Square Park|       60.00|             3.81|2025-04-29 14:41:...|
+--------+-------------------+------------+-----------------+--------------------+



In [101]:
# Procesar datos para lugares
df_venues_p = df_venues.select(
    col("venue_id").cast("long").alias("venue_id"),
    col("address_1").alias("venue_address"),
    col("city").alias("venue_city"),
    #col("state").alias("venue_state"),
    #col("zip").alias("venue_zip"),
    col("country").alias("venue_country"),
    #col("localized_country_name").alias("venue_localized_country_name"),
    #col("lat").alias("venue_lat"),
    #col("lon").alias("venue_lon"),
    #col("distance").alias("venue_distance"),
    col("venue_name"),
    #col("rating").alias("venue_rating"),
    col("rating_count").alias("venue_rating_count"),
    col("normalised_rating").alias("venue_normalised_rating"),
    col("load_timestamp")
)

print(df_venues_p.count(), len(df_venues_p.columns))
df_venues_p.show(5)


107093 8
+--------+-------------------+----------+-------------+--------------------+------------------+-----------------------+--------------------+
|venue_id|      venue_address|venue_city|venue_country|          venue_name|venue_rating_count|venue_normalised_rating|      load_timestamp|
+--------+-------------------+----------+-------------+--------------------+------------------+-----------------------+--------------------+
|    5286|     424 Park Ave S|  New York|           us|    Starbucks Coffee|             51.00|                   3.92|2025-04-29 14:43:...|
|    5293|       Union Square|  New York|           us|Virgin Megastore ...|            109.00|                   2.80|2025-04-29 14:43:...|
|    8356|  141 West 72nd St.|  New York|           us|Krispy Kreme Doug...|             18.00|                   2.00|2025-04-29 14:43:...|
|   11898|7000 North Glenwood|   Chicago|           us|      Heartland Cafe|            275.00|                   3.79|2025-04-29 14:43:...|
|   

In [113]:
# 1. Guardar el DataFrame como CSV en el sistema de archivos local
df_venues_p.write.csv("meetup_venues_processed", header=True, mode="overwrite")

# 2. Comprimir los resultados (PySpark genera múltiples archivos)
!zip -r meetup_data.zip meetup_venues_processed/

# 3. Descargar el archivo comprimido (en Colab)
from google.colab import files
files.download("meetup_data.zip")

updating: meetup_venues_processed/ (stored 0%)
updating: meetup_venues_processed/_SUCCESS (stored 0%)
updating: meetup_venues_processed/._SUCCESS.crc (stored 0%)
  adding: meetup_venues_processed/.part-00000-9772d50c-155d-4996-8418-e08b722a0916-c000.csv.crc (deflated 0%)
  adding: meetup_venues_processed/part-00000-9772d50c-155d-4996-8418-e08b722a0916-c000.csv (deflated 75%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [106]:
df_venues.coalesce(1) \
    .write \
    .format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save("meetup_venues_single_file")

In [109]:
import os
import shutil
from IPython.display import display, FileLink

# 1. Guardar el DataFrame como un único archivo CSV
output_dir = "meetup_venues_single_file"
df.coalesce(1).write.csv(output_dir, header=True, mode="overwrite")

# 2. Buscar el archivo generado (el que comienza con 'part-00000')
for file in os.listdir(output_dir):
    if file.startswith('part-00000'):
        csv_file = os.path.join(output_dir, file)
        break

# 3. Renombrar el archivo (opcional pero recomendado)
final_filename = "venues_consolidado.csv"
shutil.move(csv_file, final_filename)

# 4. Crear enlace de descarga clickeable
display(FileLink(final_filename))  # Esto mostrará un link en tu notebook

# 5. (Opcional) Eliminar el directorio temporal
shutil.rmtree(output_dir)

/content/venues_consolidado.csv

In [110]:
from google.colab import files

# Genera el archivo (con el código anterior)
files.download("venues_consolidado.csv")  # Descarga automática

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [111]:
df_venues_p.printSchema()

root
 |-- venue_id: long (nullable = true)
 |-- venue_address: string (nullable = true)
 |-- venue_city: string (nullable = true)
 |-- venue_country: string (nullable = true)
 |-- venue_name: string (nullable = true)
 |-- venue_rating_count: string (nullable = true)
 |-- venue_normalised_rating: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = false)

